# 🔬 Notebook 3: Airbnb — Deep Dive

## 🛠️ Setup

```bash
cd 06-system-designs/airbnb
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1

### Preventing double bookings

When two guests try to book the same dates simultaneously, naive checks fail:

```
T1: SELECT availability → free
T2: SELECT availability → free
T1: INSERT booking  ✅
T2: INSERT booking  ✅ ← DOUBLE BOOKING!
```

Fix: wrap in a **transaction** with `SELECT ... FOR UPDATE` (pessimistic lock) or use a **unique constraint** on (listing_id, date) in an availability table so the DB itself rejects the second insert.

In [ ]:
import threading

class Calendar:
    def __init__(self):
        self._booked = set()        # set of (listing_id, day)
        self._lock = threading.Lock()

    def book(self, listing_id, days):
        # days: list of date objects
        with self._lock:
            keys = [(listing_id, d) for d in days]
            if any(k in self._booked for k in keys):
                return False
            self._booked.update(keys)
            return True

from datetime import date, timedelta
cal = Calendar()
days = [date(2026,5,1) + timedelta(days=i) for i in range(3)]

def try_book(n):
    ok = cal.book(1, days)
    print(f"thread {n}: {'OK' if ok else 'REJECTED'}")

ts = [threading.Thread(target=try_book, args=(i,)) for i in range(5)]
for t in ts: t.start()
for t in ts: t.join()
# Exactly one succeeds.

## Deep dive 2

### Geo search with a grid index

Scanning 10M listings for each query is too slow. Two common indexes:

- **Geohash** — encode (lat,lng) into a string; nearby points share a prefix.
- **S2 / H3** — hierarchical cells.

Below, a toy geohash bucket index: we put each listing in a coarse grid bucket and only scan buckets near the query.

In [ ]:
from collections import defaultdict

def bucket(lat, lng, precision=1):
    return (round(lat, precision), round(lng, precision))

index = defaultdict(list)
listings = [
    (1, 47.60, -122.33),
    (2, 47.61, -122.30),
    (3, 40.71, -74.00),   # NYC
    (4, 34.05, -118.24),  # LA
]
for lid, lat, lng in listings:
    index[bucket(lat, lng)].append(lid)

def nearby(lat, lng, precision=1):
    # Check the target bucket + 8 neighbors
    results = []
    step = 10 ** -precision
    for dlat in (-step, 0, step):
        for dlng in (-step, 0, step):
            results += index.get(bucket(lat+dlat, lng+dlng, precision), [])
    return results

print("near Seattle:", nearby(47.60, -122.33))
print("near NYC:", nearby(40.71, -74.00))

## Closing thoughts

- The **booking** problem is a classic serial-write bottleneck — solved with DB locks or unique constraints.
- The **search** problem is a read-scaling problem — solved with a dedicated search index.
- Cache the hottest listing pages (top-of-search) behind a CDN/edge cache.